# Which dataset characteristics actually matter

Stage 2f. Goodness-of-fit was shown against every statistical characteristic as
a rolling average over 250 datasets each side, in a 21-panel figure per arm.
That presentation carries no uncertainty band, shows no data density, and gives
21 MARGINAL views of characteristics that are correlated with each other, so it
overstates how many independent effects there are.

**This notebook is separate from the other three on purpose.** It reads the
characteristics from notebook 1, the goodness-of-fit and cross-validated scores
from notebook 2, and the run against the true parent distributions from
notebook 3, and it computes nothing that needs the probabilistic LCA machinery.
It runs in about fifteen minutes, which is what makes the reduction something
that can be iterated on.

**THERE ARE THREE TARGETS AND THEY ANSWER DIFFERENT QUESTIONS.**

| target | the question | where the answer lives |
|---|---|---|
| the LEVEL of a method's own score | how far is this fitted curve from its target | dominated by dispersion and dataset size, because every method gets worse on spread, small data |
| the CHOICE between two families | which method should I use | whose shape assumption fits: skewness, kurtosis, lognormality, outlier weight |
| the ANSWER a pLCA gives | how wrong is the final number | between the two, and closest to size |

Reporting the LEVEL ranking as an answer to the CHOICE question is a mistake
this analysis made in its first pass, and the two are now separated.

In [ ]:
import json
import os
import sys
import time
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from itertools import product

sys.path.insert(0, '../src')

# Drop this project's own modules before importing them, so a kernel that has
# been open across an edit picks up the edit.
for _name in [_n for _n in list(sys.modules)
              if _n in ('comparison', 'corpus', 'coverage', 'customstats',
                        'empirical', 'families', 'figstyle', 'fitting',
                        'genconfig', 'metricreduction', 'modality')]:
    del sys.modules[_name]

import comparison
import corpus
import coverage as coverage_mod
import figstyle
import metricreduction as metricred

# Screen resolution for the figure, print resolution for the file.
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['savefig.dpi'] = 300

OUT = '../outputs'

# All randomness in this notebook comes from this Generator.
SEED = 42
rng = np.random.default_rng(SEED)

PE = ['Normal', 'Lognormal', 'KDE']
WT = ['Uniform', 'Variable']
PEWT = [f'{pe}, {wt}' for pe, wt in product(PE, WT)]
dct_colors = {pewt: c for pewt, c
              in zip(PEWT, sns.color_palette('Paired', len(PEWT)))}
with open('../src/dct_metriclabels.json') as f:
    dct_metriclabels = json.load(f)
print(f'{len(PEWT)} methods, seed {SEED}')

In [ ]:
# THE THREE INPUTS, each written by an earlier notebook, and the candidate set.
#
# 25 candidates: the 21 characteristics the old figure drew, the two
# visible-mode counts, and the author's modality index at the bandwidth the
# study FITS -- which is the only version of that measure never tested, the
# existing column being the same index at Scott's rule.
_emp = pd.read_excel(f'{OUT}/tables/TABLE_EmpiricalECCMetrics.xlsx',
                     index_col=0).reset_index()
_emp.columns = ['dataset'] + list(_emp.columns[1:])
_emp['arm'] = 'empirical'

_metrics_all, _, _runmeta = corpus.load_corpus(with_values=False)
_syn = _metrics_all[~_metrics_all.is_probe].copy()
_syn['arm'] = 'synthetic'
_shared = [c for c in _emp.columns if c in _syn.columns]
red_chars = pd.concat([_emp[_shared], _syn[_shared]], ignore_index=True)

red_scores = pd.read_csv(f'{OUT}/tables/TABLE_TargetComparison.csv')
red_modes = pd.read_csv(f'{OUT}/tables/TABLE_VisibleModes.csv')
red_truth = pd.read_csv(f'{OUT}/tables/TABLE_PLCATruth.csv.gz')

red_frame, red_metrics = metricred.assemble(red_chars, red_scores,
                                            truth=red_truth, modes=red_modes)
red_metrics = list(red_metrics) + [
    m for m in metricred.MODE_METRICS + metricred.EXTRA_METRICS
    if m in red_frame.columns]

print(f"corpus {_runmeta['label']}  "
      f"{len(red_frame):,} rows over {red_frame.dataset.nunique():,} datasets")
print(f'{len(red_metrics)} candidate characteristics')
for _t in ('w1', 'w1_cv', 'w1_market', 'err_eci_mean', 'err_eci_rank_1',
           'err_ui', 'err_eci_p95'):
    if _t in red_frame:
        _c = red_frame.groupby('arm')[_t].apply(lambda s: int(s.notna().sum()))
        print(f'  {_t:16s} ' + '  '.join(f'{k} {v:,}' for k, v in _c.items()))

## What the models are allowed to forget

In [ ]:
# WHAT A COMPLETE-CASE MODEL WOULD THROW AWAY, before any model is fitted.
#
# Two different things remove the smallest datasets and neither announces
# itself. Excess kurtosis divides by (n-1)(n-2)(n-3) so it is undefined below
# four values; a visible mode count needs eight. Both models below keep every
# row -- the additive one imputes with a missingness indicator, the boosted one
# splits on missingness natively -- and these tables are what say why that
# mattered.
red_missing = metricred.missingness_by_band(red_frame, red_metrics)
red_missing.to_csv(f'{OUT}/tables/TABLE_ReductionMissingness.csv', index=False)
red_cost = metricred.complete_case_cost(red_frame, red_metrics)
red_cost.to_csv(f'{OUT}/tables/TABLE_ReductionCompleteCaseCost.csv',
                index=False)
display(red_cost)

# The SECOND exclusion, which the table above cannot see: the cross-validated
# empirical target is undefined below n = 10, because half of a nine-value
# dataset is four values. So the out-of-sample reduction on the real arm has
# nothing to say about the smallest band, and those claims rest on the
# in-sample target instead.
red_used = metricred.rows_used_by_band(
    red_frame, ('w1', 'w1_cv', 'w1_market', 'err_eci_mean', 'err_eci_rank_1',
                'err_ui', 'err_eci_p95'))
red_used.to_csv(f'{OUT}/tables/TABLE_ReductionRowsUsed.csv', index=False)
display(red_used[red_used.method == 'KDE, Uniform']
        [['arm', 'target', 'rows_total', 'rows_used']
         + [c for c in red_used.columns if c.startswith('used__')]])

## How many independent things are there, and what are they

In [ ]:
# THE CORRELATION STRUCTURE, and then the DIRECTIONS.
#
# The effective dimension is the participation ratio of the correlation
# eigenvalues: it would be 25 if the candidates were independent and 1 if they
# were all the same quantity. It says HOW MANY independent quantities there
# are and not WHAT they are, which is what the components below are for --
# "about four or five" is not something a reader can act on.
red_redundancy = metricred.redundancy_table(red_frame, red_metrics)
red_redundancy.to_csv(f'{OUT}/tables/TABLE_ReductionRedundancy.csv',
                      index=False)
print('EFFECTIVE DIMENSION of the candidate set')
print(red_redundancy.groupby('arm')[['effective_dimension', 'n_metrics']]
      .first().to_string())
print()
print('the most redundant pairs -- a pair above 0.9 is one quantity')
display(red_redundancy.head(10)[['arm', 'metric_a', 'metric_b', 'correlation']])

red_pca = metricred.principal_components(red_frame, red_metrics)
red_pca.to_csv(f'{OUT}/tables/TABLE_ReductionComponents.csv', index=False)
for _arm in ('empirical', 'synthetic'):
    _g = red_pca[red_pca.arm == _arm]
    if not len(_g):
        continue
    print(f'\n{_arm}: {_g.n_datasets.iloc[0]:,} complete datasets, '
          f'{_g.n_characteristics.iloc[0]} characteristics, '
          f'participation ratio {_g.participation_ratio.iloc[0]:.2f}')
    for _, r in _g.iterrows():
        print(f'  {r.component}  {r.variance_share*100:4.1f} pct '
              f'(cum {r.cumulative_share*100:4.1f})  {r.loadings}')

## The question the paper asks: which method should I use

The reduction's first pass ranked characteristics by how well they predict the
LEVEL of a single method's score. The level is dominated by dispersion and
dataset size, because every one of the six methods gets worse on spread data
and on small samples. **That is not the question the paper asks.** The paper
asks which method to use, which is the DIFFERENCE between two of them, and a
difference is about whose shape assumption fits.

Two properties of the ratio matter. Taken within a weighting scheme it CANCELS
the part of the score no estimator can remove, which makes the
uniform-to-variable distance a legitimate predictor here where on the level it
was an identity. And it is scale free, so it does not inherit the level's
dependence on how spread the data happen to be.

In [ ]:
# WHAT EACH CHARACTERISTIC ADDS TO THE CHOICE, MEASURED OUT OF SAMPLE.
#
# THE SYNTHETIC ARM IS PRIMARY AND THE REAL ARM IS A CHECK. That is not a
# preference, it is what the numbers force. Cross-validated on the 127 real
# categories the base model of size and dispersion reaches an R2 of -0.724
# under market-share weights -- it predicts WORSE THAN THE MEAN -- and only 9
# of 46 candidate gains exceed their own fold-to-fold spread. An in-sample R2
# on that arm rises by about 0.043 just from adding five spline terms, which
# is most of what the earlier version of this cell reported as signal.
#
# NO P-VALUES. A p-value answers "could this be zero", which is not the
# question; the question is how big the gain is and how firm. Every gain below
# is reported beside the standard deviation of that gain across folds, and a
# gain smaller than its own fold spread is not a finding however small its
# p-value would have been.
red_choice = {}
for _value, _arm in (('w1_cv', 'empirical'), ('w1_market', 'synthetic')):
    red_choice[_arm] = metricred.choice_frame(red_frame, _value)

_rows = []
for _arm, _ch in red_choice.items():
    for _wt in ('Uniform', 'Variable'):
        _d = metricred.cv_gain(_ch, red_metrics, base=('n', 'coeffvar'),
                               arm=_arm, weighting=_wt, seed=SEED)
        if len(_d):
            _rows.append(_d)
red_choice_cv = pd.concat(_rows, ignore_index=True)
red_choice_cv.to_csv(f'{OUT}/tables/TABLE_ReductionChoiceCV.csv', index=False)

for _arm in ('synthetic', 'empirical'):
    for _wt in ('Uniform', 'Variable'):
        _g = red_choice_cv[(red_choice_cv.arm == _arm)
                           & (red_choice_cv.weighting == _wt)]
        if not len(_g):
            continue
        _base = _g.base_cv_r2.iloc[0]
        _flag = '  <-- predicts worse than the mean' if _base < 0 else ''
        print(f'{_arm} / {_wt}: {_g.n_rows.max():,} datasets, base '
              f'out-of-sample R2 {_base:+.3f}{_flag}')
        print(f'   gains above their own fold spread: '
              f'{int(_g.exceeds_fold_spread.sum())} of {len(_g)}')
        display(_g.head(6)[['metric', 'cv_gain', 'fold_sd',
                            'exceeds_fold_spread']].round(4))


In [ ]:
# THE SAME QUESTION BY PERMUTATION IMPORTANCE, on both model families, so the
# ranking does not rest on one modeling assumption.
_ci = []
for _arm, _ch in red_choice.items():
    for _wt in ('Uniform', 'Variable'):
        _d = metricred.choice_importance(_ch, red_metrics, arm=_arm,
                                         weighting=_wt, rng=rng)
        if len(_d):
            _ci.append(_d)
red_choice_importance = pd.concat(_ci, ignore_index=True)
red_choice_importance.to_csv(
    f'{OUT}/tables/TABLE_ReductionChoiceImportance.csv', index=False)
print('out-of-sample R2 of the choice models themselves; an importance inside')
print('a model that predicts nothing means nothing')
display(red_choice_importance.groupby(['arm', 'weighting', 'model'])
        .model_r2.first().unstack().round(3))
print('top predictors of WHICH METHOD WINS, averaged over the models above')
display(red_choice_importance.groupby('metric')
        .agg(mean_importance=('importance', 'mean'),
             max_importance=('importance', 'max'))
        .sort_values('mean_importance', ascending=False).head(10).round(4))

## Redundancy, and the thresholds that follow

In [ ]:
# IS THE OUT-OF-SAMPLE R2 ITSELF A NUMBER? On one arm it is not.
#
# The base model is the same two predictors on the same datasets every time;
# only which datasets land in which fold changes. On the corpus that changes
# nothing. On 127 real categories it moves the answer by three points, so any
# single value quoted from that arm -- including a flattering one -- is a draw
# from a distribution rather than a measurement. This is the cell that settles
# why the reduction is reported on the synthetic arm.
_rows = []
for _arm, _ch in red_choice.items():
    for _wt in ('Uniform', 'Variable'):
        _g = _ch[(_ch.arm == _arm) & (_ch.weighting == _wt)]
        _M = metricred.transform_frame(_g, ['n', 'coeffvar'])
        _y = _g.log_ratio.to_numpy(float)
        _r = np.array([metricred.cv_r2(_M, ['n', 'coeffvar'], _y, seed=_s)[0]
                       for _s in range(20)])
        _rows.append(dict(arm=_arm, weighting=_wt, n_datasets=len(_g),
                          base_r2_median=float(np.median(_r)),
                          base_r2_min=float(_r.min()),
                          base_r2_max=float(_r.max()),
                          base_r2_sd=float(_r.std())))
red_base_stability = pd.DataFrame(_rows)
red_base_stability.to_csv(f'{OUT}/tables/TABLE_ReductionBaseStability.csv',
                          index=False)
display(red_base_stability.round(3))
print('the corpus reproduces its own R2 to three decimals across every fold')
print('assignment; the real arm does not reproduce its own SIGN')


In [ ]:
# WHICH CHARACTERISTICS SURVIVE ONCE THE OTHERS ARE ALREADY IN THE MODEL.
#
# THIS IS THE REDUNDANCY CONTROL AND IT CHANGES THE ANSWER. Skewness, kurtosis
# and the two normality statistics are four views of one shape, so a
# one-at-a-time table credits the same effect four times and reading its top
# five as "the five that matter" overcounts. Forward selection adds a
# characteristic only if it still improves out-of-sample prediction once
# everything already chosen is present.
_sel = []
for _arm, _ch in red_choice.items():
    for _wt in ('Uniform', 'Variable'):
        _s = metricred.forward_select(_ch, red_metrics, base=('n', 'coeffvar'),
                                      arm=_arm, weighting=_wt, seed=SEED)
        if len(_s):
            _sel.append(_s)
red_choice_selection = pd.concat(_sel, ignore_index=True)
red_choice_selection.to_csv(
    f'{OUT}/tables/TABLE_ReductionChoiceSelection.csv', index=False)

for (_arm, _wt), _g in red_choice_selection.groupby(['arm', 'weighting']):
    print(f'{_arm} / {_wt}: R2 {_g.base_cv_r2.iloc[0]:+.3f} -> '
          f'{_g.cv_r2.iloc[-1]:+.3f}')
    print('   kept, in order: ' + ', '.join(
        f'{r.added} (+{r.gain:.3f})' for r in _g.itertuples()))
display(red_choice_selection)


In [ ]:
# THE THRESHOLDS A PRACTITIONER CAN ACT ON.
#
# The reduction says which characteristics carry signal. That is not yet
# advice. These three tables turn it into the statements the study exists to
# make: is one method best regardless, where is the dataset-size cutoff, and
# when does market-share weighting pay.
#
# ALL THREE ARE SCORED AGAINST THE MARKET-WEIGHTED PARENT, and the choice of
# target is doing real work. Under `w1_parent` each weighting scheme is graded
# against a DIFFERENT population, so a comparison across schemes is
# meaningless; under the in-sample `w1` every model is scored against the
# variable-weighted data, which makes market-share weighting win by
# construction. Only `w1_market` puts all six on one footing.
_sizes = _metrics_all.set_index('dataset')['n']

red_best_share = metricred.best_method_share(red_scores, _sizes,
                                             value='w1_market')
red_best_share.to_csv(f'{OUT}/tables/TABLE_ReductionBestMethod.csv',
                      index=False)

# the same thing without the bins, which is what the figure draws
red_best_curve = metricred.best_method_curve(red_scores, _sizes,
                                             value='w1_market')
red_best_curve.to_csv(f'{OUT}/tables/TABLE_ReductionBestMethodCurve.csv',
                      index=False)
print('share of datasets on which each method is closest to the truth')
display(red_best_share.round(1))

red_policy = metricred.policy_curve(red_scores, _sizes, 'KDE, Variable',
                                    'Lognormal, Uniform', value='w1_market')
red_policy.to_csv(f'{OUT}/tables/TABLE_ReductionPolicyCurve.csv', index=False)
red_flat = metricred.flat_region(red_policy)
print(f"\ncutoff: best at n = {red_flat['best_threshold']}, and every "
      f"threshold from {red_flat['lo']} to {red_flat['hi']} is as good")
print(f"   cost over the per-dataset oracle {red_flat['best_cost']:.1f} pct, "
      f"against {red_flat['worst_fixed_policy_cost']:.1f} pct for the better "
      f"of the two fixed policies")

# Why market-share weighting pays, and it is NOT that it shrinks the sample.
_, _values_all, _ = corpus.load_corpus()
red_eff = metricred.effective_sample_fraction(_values_all)
_conc = [metricred.weighting_by_concentration(red_scores, red_eff, family=_f,
                                              n_lo=_lo, n_hi=_hi)
         for _f in ('KDE', 'Lognormal')
         for _lo, _hi in ((3, 9), (100, 999))]
red_concentration = pd.concat([c for c in _conc if len(c)], ignore_index=True)
red_concentration.to_csv(f'{OUT}/tables/TABLE_ReductionWeightConcentration.csv',
                         index=False)
print('\nhow often the market-share fit beats its own uniform twin, by how')
print('concentrated the weights are (q1 = most concentrated)')
display(red_concentration.round(3))
del _values_all


## The level and the answer, for comparison

Kept because the contrast between the three targets is the finding. The level
is what the figure this notebook replaces was drawn against.

In [ ]:
# THE LEVEL AND THE ANSWER TARGETS. This is the expensive cell, about ten
# minutes: roughly fifty (arm, method, target) cells, two models each, each
# ranked by permutation importance on the held-out fold of a five-fold split.
_t0 = time.time()
FIT_TARGETS = ('w1', 'w1_cv', 'w1_market')
ANSWER_TARGETS = ('err_eci_mean', 'err_eci_rank_1', 'err_ui', 'err_eci_p95')

_plan = []
for _t in FIT_TARGETS:
    for _arm in ('empirical', 'synthetic'):
        _plan += [(_arm, _m, _t) for _m in PEWT]
for _t in ANSWER_TARGETS:
    _plan += [('synthetic', _m, _t) for _m in PEWT]

_imps = []
for _arm, _method, _target in _plan:
    if _target not in red_frame.columns:
        continue
    _sub = red_frame[(red_frame.arm == _arm) & (red_frame.method == _method)]
    if _sub[_target].notna().sum() < 60:
        continue
    _imps.append(metricred.importance(red_frame, red_metrics, _target,
                                      arm=_arm, method=_method, rng=rng))
red_importance = pd.concat([d for d in _imps if len(d)], ignore_index=True)
red_importance['target_family'] = red_importance.target.map(
    lambda t: metricred.TARGETS[t]['family'])
red_importance.to_csv(f'{OUT}/tables/TABLE_ReductionImportance.csv',
                      index=False)
print(f'{len(red_importance):,} importance rows in {time.time()-_t0:.0f}s')
display(red_importance.groupby(['target', 'arm', 'model'])
        .model_r2.first().unstack().round(3))

In [ ]:
# IS A CANDIDATE PREDICTING A TARGET, OR IS IT PART OF ONE?
#
# A fit score splits into the part an estimator can remove and a definitional
# part it cannot, and for an equal-weighted method that definitional part IS
# the uniform-to-variable distance, because every model is scored against the
# market-weighted target. Ranking it first on a LEVEL target rediscovers an
# identity. On the CHOICE target, taken within a weighting scheme, the
# identity cancels from both terms and it becomes a real predictor.
red_definitional = metricred.definitional_check(red_frame, red_scores,
                                                metrics=red_metrics)
red_definitional.to_csv(f'{OUT}/tables/TABLE_ReductionDefinitional.csv',
                        index=False)
_ident = red_definitional[red_definitional.is_an_identity]
display(_ident[['arm', 'method', 'metric', 'spearman_definitional',
                'spearman_w1', 'spearman_err_eci_mean']].round(4))

In [ ]:
# IS A CHARACTERISTIC JUST DATASET SIZE WEARING ANOTHER NAME?
red_size = metricred.size_confounding(red_frame, red_metrics)
red_size.to_csv(f'{OUT}/tables/TABLE_ReductionSizeConfounding.csv', index=False)
for _arm in ('empirical', 'synthetic'):
    print(f'{_arm}: most explained by log(n) alone')
    display(red_size[red_size.arm == _arm].head(5)
            [['metric', 'n_used', 'r2_on_log_n', 'spearman_with_log_n']])

In [ ]:
# THE SURVIVORS, by target family, with and without the definitional
# candidate. A metric's score is its mean rank across every model that
# predicted anything at all, and a model whose out-of-sample R2 is below 0.02
# contributes nothing because inside it the ordering is noise.
_all_imp = pd.concat([red_importance, red_choice_importance], ignore_index=True)
_surv = []
for _fam in ('fit', 'answer', 'choice'):
    for _trim, _tag in ((False, ''), (True, ', definitional removed')):
        _sub = _all_imp[_all_imp.target_family == _fam]
        if _trim:
            _sub = _sub[~_sub.metric.isin(metricred.DEFINITIONAL_CANDIDATES)]
        _s = metricred.rank_survivors(_sub)
        if len(_s):
            _s.insert(0, 'target_family', _fam + _tag)
            _surv.append(_s)
red_survivors = pd.concat(_surv, ignore_index=True)
red_survivors.to_csv(f'{OUT}/tables/TABLE_ReductionSurvivors.csv', index=False)
for _fam in red_survivors.target_family.unique():
    _s = red_survivors[red_survivors.target_family == _fam]
    print(_fam.upper())
    display(_s.head(7)[['metric', 'mean_rank', 'mean_importance',
                        'share_top5', 'n_models', 'survivor']])

SURVIVORS_CHOICE = list(
    red_survivors[red_survivors.target_family == 'choice, definitional removed']
    .query('survivor').metric)
print('choice survivors carried into the figures:', ', '.join(SURVIVORS_CHOICE))

In [ ]:
# WHICH CHARACTERISTICS SURVIVE ONE TARGET AND NOT ANOTHER. These are the
# interesting ones, and the level-versus-choice column is the one that
# corrects the reduction's first pass.
_ranks = (red_survivors[red_survivors.target_family.str.endswith(
              'definitional removed')]
          .assign(fam=lambda d: d.target_family.str.split(',').str[0])
          .pivot_table(index='metric', columns='fam', values='mean_rank'))
_ranks['choice_minus_fit'] = _ranks.get('choice') - _ranks.get('fit')
_ranks['answer_minus_fit'] = _ranks.get('answer') - _ranks.get('fit')
red_target_shift = _ranks.sort_values('choice_minus_fit').reset_index()
red_target_shift.to_csv(f'{OUT}/tables/TABLE_ReductionTargetShift.csv',
                        index=False)
print('negative choice_minus_fit = matters MORE for the CHOICE than for the fit')
display(red_target_shift.round(2))

## Post-stratification, modality, and coverage

In [ ]:
# POST-STRATIFIED: the same reduction on a corpus resampled to the EMPIRICAL
# size mix, because the corpus allocates datasets equally across four size
# bands for precision while the real categories are nothing like that.
_shares = metricred.empirical_size_shares(red_frame)
print('empirical size mix: ' + ', '.join(f'{k} {v:.3f}'
                                         for k, v in _shares.items()))
red_ps_frame = metricred.size_mix_resample(red_frame, _shares, rng=rng)
print(f'{red_ps_frame.dataset.nunique():,} synthetic datasets at that mix')
_ps = []
for _target in ('w1_market', 'err_eci_mean'):
    for _method in PEWT:
        _d = metricred.importance(red_ps_frame, red_metrics, _target,
                                  arm='synthetic', method=_method, rng=rng)
        if len(_d):
            _ps.append(_d)
red_importance_ps = pd.concat(_ps, ignore_index=True)
red_importance_ps['allocation'] = 'empirical size mix'
_eq = red_importance[(red_importance.arm == 'synthetic')
                     & red_importance.target.isin(('w1_market',
                                                   'err_eci_mean'))].copy()
_eq['allocation'] = 'equal'
red_poststrat = pd.concat([_eq, red_importance_ps], ignore_index=True)
red_poststrat.to_csv(f'{OUT}/tables/TABLE_ReductionPostStratified.csv',
                     index=False)
display(red_poststrat.groupby(['allocation', 'metric']).importance.mean()
        .unstack(0).sort_values('equal', ascending=False).head(10).round(4))

In [ ]:
# THE MODALITY MEASURES, HEAD TO HEAD, and there are now four of them.
#
# `modality_index` is the author's index off a SCOTT bandwidth, which is what
# it was written with and what the study stopped using in Stage 2b.
# `modality_index_fitted` is the same index at the bandwidth the study FITS,
# which is the only version never tested before Stage 2f, and the difference
# is not cosmetic: at Scott's bandwidth the index spans 1.000 to 1.074 across
# the real categories, which is what Stage 2a read as "constant at 1" when it
# set the measure aside, and at the fitted bandwidth it spans 1.000 to 1.249.
# `crit_bw_1` is Silverman's critical bandwidth and `modes_fitted` is a count.
red_mod_agree = metricred.modality_agreement(red_frame)
red_mod_agree.to_csv(f'{OUT}/tables/TABLE_ReductionModalityAgreement.csv',
                     index=False)
display(red_mod_agree[red_mod_agree.measure_b != 'share_unimodal'].round(3))

_mh = []
for _arm, _targets in (('empirical', ('w1', 'w1_cv')),
                       ('synthetic', ('w1', 'w1_market', 'err_eci_mean'))):
    _d = metricred.modality_head_to_head(red_frame, _targets, PEWT, _arm,
                                         rng=rng)
    if len(_d):
        _mh.append(_d)
red_modality = pd.concat(_mh, ignore_index=True)
red_modality.to_csv(f'{OUT}/tables/TABLE_ReductionModality.csv', index=False)
print('what each modality measure adds over a spline in log(n) ALONE')
display(red_modality.round(4))

In [ ]:
# DOES THE CORPUS COVER THE CHARACTERISTICS THAT TURN OUT TO MATTER?
#
# The generator's tuning objective matches the SHAPE of the synthetic
# characteristic distribution to the empirical one, while the study also needs
# to SPAN that space with margin so its conclusions generalize past the
# categories EC3 happens to hold. If the characteristics that predict are the
# ones the corpus reaches least far on, the generalization claim is narrower
# than the coverage figure suggests.
_cov = coverage_mod.coverage_table(
    red_chars[red_chars.arm == 'empirical'],
    red_chars[red_chars.arm == 'synthetic'],
    metrics=tuple(m for m in red_metrics if m in red_chars.columns))
_pooled = metricred.rank_survivors(
    _all_imp[~_all_imp.metric.isin(metricred.DEFINITIONAL_CANDIDATES)])
red_cov_imp = metricred.coverage_versus_importance(_cov, _pooled)
red_cov_imp.to_csv(f'{OUT}/tables/TABLE_ReductionCoverageVsImportance.csv',
                   index=False)
display(red_cov_imp.round(4))
for _k, _v in red_cov_imp.attrs.items():
    print(f'{_k}: {_v:+.3f}')

# AND IN REAL UNITS, because "margin of -0.63" is not actionable and "the
# corpus reaches 2.6 where the data reach 6.9" is.
_short = _cov[_cov.margin_above < 0].copy()
_short['short_by'] = _short.empirical_max / _short.synthetic_max
red_shortfall = _short[['metric', 'empirical_max', 'synthetic_max',
                        'short_by', 'empirical_covered']]
red_shortfall.to_csv(f'{OUT}/tables/TABLE_ReductionCoverageShortfall.csv',
                     index=False)
print('\nWHERE THE CORPUS DOES NOT REACH THE REAL MAXIMUM')
display(red_shortfall.round(3))

## The figures

In [ ]:
# THE CURVES THAT REPLACE THE ROLLING AVERAGES, as a table on disk.
#
# Equal-count bins with a within-bin percentile bootstrap carry the
# uncertainty and the counts; a LOWESS smooth bounded to the span the bins
# cover is the read through them.
_curves = []
for _arm, _values in (('empirical', ('w1', 'w1_cv')),
                      ('synthetic', ('w1', 'w1_market', 'err_eci_mean'))):
    for _value in _values:
        if _value not in red_frame.columns:
            continue
        if red_frame.loc[red_frame.arm == _arm, _value].notna().sum() < 60:
            continue
        _c = metricred.curves_for(red_frame, SURVIVORS_CHOICE, list(PEWT),
                                  value=_value, arm=_arm, rng=rng)
        if len(_c):
            _c['value'] = _value
            _curves.append(_c)
red_curves = pd.concat(_curves, ignore_index=True)
red_curves.to_csv(f'{OUT}/tables/TABLE_ReductionCurves.csv.gz', index=False,
                  compression='gzip')
print(f'{len(red_curves):,} curve rows over '
      f'{red_curves.characteristic.nunique()} characteristics')

In [ ]:
# THE SIZE CURVE AS A TABLE, so the figure is drawn from disk.
#
# Equal-count bins, so every point rests on the same number of datasets and a
# sparse tail cannot be mistaken for a trend. The band is a bootstrap over
# datasets within each bin.
_rows = []
for _arm, _ch in red_choice.items():
    for _wt in ('Uniform', 'Variable'):
        _g = _ch[_ch.weighting == _wt].dropna(subset=['n', 'log_ratio'])
        if len(_g) < 100:
            continue
        _nb = 10 if _arm == 'synthetic' else 5
        _q = np.unique(np.quantile(_g.n, np.linspace(0, 1, _nb + 1)))
        _idx = np.digitize(_g.n, _q[1:-1])
        for _b in range(len(_q) - 1):
            _m = _idx == _b
            if _m.sum() < 20:
                continue
            _w = (_g.log_ratio.to_numpy()[_m] < 0).astype(float)
            _bs = [_w[rng.integers(0, len(_w), len(_w))].mean()
                   for _ in range(1000)]
            _rows.append(dict(
                arm=_arm, weighting=_wt, n_median=float(np.median(_g.n[_m])),
                n_lo=float(_g.n[_m].min()), n_hi=float(_g.n[_m].max()),
                kde_closer_pct=float(100 * _w.mean()),
                lo95=float(100 * np.quantile(_bs, .025)),
                hi95=float(100 * np.quantile(_bs, .975)),
                n_datasets=int(_m.sum())))
red_size_curve = pd.DataFrame(_rows)
red_size_curve.to_csv(f'{OUT}/tables/TABLE_ReductionSizeCurve.csv', index=False)
display(red_size_curve[red_size_curve.arm == 'synthetic'].round(1))


In [ ]:
# THE SELECTION PATH FROM NOTHING, so dataset size and dispersion are measured
# on the same axis as every other characteristic.
#
# The gain table above starts from a model that ALREADY contains size and
# dispersion, which is the right way to ask "what else matters" and the wrong
# way to draw a figure: it leaves the two biggest effects off the chart and
# then magnifies the remainder to fill it. Starting from an empty model and
# adding one characteristic at a time puts everything in the same units --
# out-of-sample R2 -- and shows the diminishing returns directly.
_paths = []
for _arm, _ch in red_choice.items():
    for _wt in ('Uniform', 'Variable'):
        _p = metricred.forward_select(_ch, red_metrics, base=(), arm=_arm,
                                      weighting=_wt, seed=SEED, max_steps=8,
                                      min_gain=0.001)
        if len(_p):
            _paths.append(_p)
red_selection_path = pd.concat(_paths, ignore_index=True)
red_selection_path.to_csv(f'{OUT}/tables/TABLE_ReductionSelectionPath.csv',
                          index=False)
for (_a, _w), _g in red_selection_path.groupby(['arm', 'weighting']):
    print(f'{_a} / {_w}: ' + ' -> '.join(
        f'{r.added} ({r.cv_r2:+.3f})' for r in _g.itertuples()))
display(red_selection_path)


In [ ]:
# WHAT EACH CHARACTERISTIC IS WORTH ON ITS OWN, which is what the figure draws.
#
# The selection path says which characteristics survive together; it does not
# say what any one of them is worth, and a reader looking at a ranked list
# wants that. This fits each candidate alone against the same target and
# reports its out-of-sample R2 with the spread across folds.
#
# THE TWO NUMBERS ANSWER DIFFERENT QUESTIONS AND BOTH BELONG IN THE FIGURE.
# Dispersion alone is worth 0.012 to 0.024 and is still KEPT, because it adds
# something dataset size does not; entropy alone is worth 0.20 and is dropped
# under market-share weights, because it is dataset size wearing another name.
# A bar chart of standalone values alone would mislead in the first direction
# and a list of survivors alone would mislead in the second.
_rows = []
for _wt in ('Uniform', 'Variable'):
    _g = red_choice['synthetic']
    _g = _g[(_g.arm == 'synthetic') & (_g.weighting == _wt)]
    _M = metricred.transform_frame(_g, red_metrics)
    _y = _g.log_ratio.to_numpy(float)
    for _m in red_metrics:
        _r2, _sd, _n = metricred.cv_r2(_M, [_m], _y, seed=SEED)
        _rows.append(dict(arm='synthetic', weighting=_wt, metric=_m,
                          alone_cv_r2=_r2, fold_sd=_sd, n_rows=_n))
red_marginal = pd.DataFrame(_rows)
_kept = red_selection_path[red_selection_path.arm == 'synthetic']
red_marginal['kept'] = [
    bool(((_kept.weighting == r.weighting) & (_kept.added == r.metric)).any())
    for r in red_marginal.itertuples()]
red_marginal.to_csv(f'{OUT}/tables/TABLE_ReductionMarginal.csv', index=False)
for _wt in ('Uniform', 'Variable'):
    _g = red_marginal[red_marginal.weighting == _wt].nlargest(6, 'alone_cv_r2')
    print(f'{_wt}: ' + ', '.join(f'{r.metric} {r.alone_cv_r2:.3f}'
                                 for r in _g.itertuples()))
display(red_marginal.sort_values(['weighting', 'alone_cv_r2'],
                                 ascending=[True, False]).head(14))


In [ ]:
# WHAT ANYTHING ADDS ONCE THE NUMBER OF DECLARATIONS IS ALREADY KNOWN.
#
# This is the question the practitioner rule turns on, and the one figure B
# draws. The other tables here ask what each characteristic is worth ALONE, or
# which survive TOGETHER; neither answers "having counted my declarations, is
# there anything else I need to look at".
_rows = []
for _wt in ('Uniform', 'Variable'):
    _rows.append(metricred.cv_gain(red_choice['synthetic'], red_metrics,
                                   base=('n',), arm='synthetic',
                                   weighting=_wt, seed=SEED))
red_over_size = pd.concat(_rows, ignore_index=True)
red_over_size.to_csv(f'{OUT}/tables/TABLE_ReductionOverSize.csv', index=False)
for _wt in ('Uniform', 'Variable'):
    _g = red_over_size[red_over_size.weighting == _wt]
    print(f'{_wt}: dataset size alone reaches R2 {_g.base_cv_r2.iloc[0]:.3f}; '
          f'best addition {_g.iloc[0].metric} at {_g.iloc[0].cv_gain:+.3f}; '
          f'nothing after the top two exceeds {_g.iloc[2:].cv_gain.max():.3f}')
display(red_over_size.head(12))


In [ ]:
# FIGURE A: which method is closest to the truth, against category size.
#
# THE MESSAGE: below about sixty declarations a lognormal is the better bet,
# the kernel estimate with market-share weights passes both lognormals at 65
# and leads outright from 107 on, and a normal fit is never the answer at any
# size.
#
# THE MARKS ARE THE CURVES' OWN CROSSINGS. An earlier version shaded 59 to 134,
# taken from the cost curve of a policy that is not drawn here -- so the band
# answered a different question from the one the panel shows, and the author
# rightly did not trust it: across most of that band the kernel estimate with
# market-share weights is already ahead of both lognormals. The two dashed
# lines are read directly off the curves above them.
#
# ALL SIX METHODS, IN THE STUDY'S OWN COLOURS, so the same method is the same
# colour in every figure and the normal's flat line along the bottom -- the
# strongest negative result here -- is visible rather than cropped out.
figstyle.apply()
_curve = pd.read_csv(f'{OUT}/tables/TABLE_ReductionBestMethodCurve.csv')
_piv = _curve.pivot_table(index='n', columns='method', values='closest_pct').sort_index()
_kv = _piv['KDE, Variable']
_rest = _piv[[c for c in _piv.columns if c != 'KDE, Variable']].max(axis=1)
_lost = np.where((_kv <= _rest).to_numpy())[0]
_lead_from = float(_piv.index[_lost.max() + 1])
_logn = _piv[['Lognormal, Uniform', 'Lognormal, Variable']].max(axis=1)
_l = np.where((_kv <= _logn).to_numpy())[0]
_beats_logn = float(_piv.index[_l.max() + 1])

fig, ax = plt.subplots(figsize=(figstyle.FULL_WIDTH * 0.74, 3.5))
_ends = {}
for _m in PEWT:
    _g = _curve[_curve.method == _m].sort_values('n')
    ax.plot(_g.n, _g.closest_pct, color=dct_colors[_m], lw=1.7)
    _ends[_m] = (_g.n.iloc[-1], _g.closest_pct.iloc[-1])
for (_m, (_x, _y)), _yy in zip(sorted(_ends.items(), key=lambda kv: kv[1][1]),
                               figstyle.stagger([v[1] for v in _ends.values()], 4.0)):
    figstyle.label_line(ax, _x, _yy, _m.replace(', ', ' '), color=dct_colors[_m], dx=5)
for _x in (_beats_logn, _lead_from):
    ax.axvline(_x, color='#999999', lw=.8, ls=(0, (4, 3)), zorder=0)
ax.annotate(f'KDE Variable passes both\nlognormals at {_beats_logn:.0f}, and leads\n'
            f'outright from {_lead_from:.0f} on',
            (_lead_from, 62), xytext=(7, 0), textcoords='offset points',
            ha='left', va='center', fontsize=figstyle.ANNOT_PT, color='#666666')
ax.set_xscale('log'); ax.set_xlim(3, 2.2e4); ax.set_ylim(-2, 88)
figstyle.finish(ax, title='Which method is closest to the truth, by category size',
                xlabel='declarations in the category',
                ylabel='closest to the truth, pct of datasets')
fig.tight_layout(); figstyle.check_overlaps(fig)
fig.savefig(f'{OUT}/figures/CompareUQMethods_FIG_WhenToUseWhich.png', bbox_inches='tight')
plt.show()


In [ ]:
# FIGURE B: once you know how many declarations there are, what else helps?
#
# THE MESSAGE: almost nothing. Under market-share weights the count alone
# reaches 0.47 and no other characteristic adds a seventh of that. The single
# large bar under equal weights is a property of the WEIGHTS rather than of the
# data -- it needs the market shares, which is exactly what a practitioner does
# not have -- so the rule stays one number.
#
# THIS REPLACES TWO FIGURES THAT ASKED THE WRONG QUESTION. One drew a selection
# path with the order along the x axis, which is not a ranking and was read as
# one. The next drew what each characteristic is worth ALONE, which left a
# reader unable to tell whether the study needs six numbers or one -- and it
# appeared to contradict the size story in figure A. Measuring each addition
# OVER dataset size answers that directly, and the two figures now agree.
figstyle.apply()
_over = pd.read_csv(f'{OUT}/tables/TABLE_ReductionOverSize.csv')
_SHORT = {'w_v_uw_wasserstein':'equal-vs-market W1','n':'dataset size',
 'coeffvar':'dispersion','coeffvar_uw':'dispersion (eq)','modes_fitted':'mode count',
 'modes_scipy_default':'mode count (Scott bw)','entropy':'entropy','entropy_uw':'entropy (eq)',
 'weight_outliers':'outlier weight','weight_outliers_uw':'outlier weight (eq)',
 'mean':'market-weighted mean','kurtosis':'kurtosis','kurtosis_uw':'kurtosis (eq)',
 'skewness':'skewness','skewness_uw':'skewness (eq)','fit_norm_SF':'normal fit',
 'fit_norm_SF_uw':'normal fit (eq)','fit_lognorm_SF':'lognormal fit',
 'fit_lognorm_SF_uw':'lognormal fit (eq)','crit_bw_1':'critical bandwidth',
 'crit_bw_1_uw':'critical bandwidth (eq)','modality_index':'modality index',
 'modality_index_fitted':'modality index (fitted bw)','modality_index_uw':'modality index (eq)',
 'modality_index_fitted_uw':'modality index (fitted bw, eq)'}
TOP = 8
fig, axes = plt.subplots(1, 2, figsize=(figstyle.FULL_WIDTH, 3.2), sharex=True)
for ax, wt, ttl in zip(axes, ('Uniform','Variable'),
                       ('Equal weights', 'Market-share weights')):
    g = _over[_over.weighting == wt].sort_values('cv_gain', ascending=False)
    base = float(g.base_cv_r2.iloc[0])
    top = g.head(TOP).iloc[::-1]
    y = np.arange(len(top))
    ax.barh(y, top.cv_gain, height=.66, color=figstyle.FAINT, zorder=3)
    for i, r in zip(y, top.itertuples()):
        ax.plot([r.cv_gain-r.fold_sd, r.cv_gain+r.fold_sd], [i,i],
                color='#444444', lw=.9, zorder=4)
    ax.set_yticks(y)
    ax.set_yticklabels([_SHORT.get(m,m) for m in top.metric],
                       fontsize=figstyle.ANNOT_PT)
    ax.axvline(base, color=figstyle.ACCENT, lw=1.6, zorder=2)
    ax.annotate(f'dataset size on its own: {base:.2f}', (base, -0.45),
                xytext=(-4, 0), textcoords='offset points', ha='right',
                va='bottom', fontsize=figstyle.ANNOT_PT, color=figstyle.ACCENT)
    ax.set_xlim(0, 0.52); ax.set_ylim(-0.9, len(top) - 0.3)
    figstyle.finish(ax, title=ttl,
                    xlabel='extra share of the variation explained')
_note = ('Bars: what each characteristic adds to predicting which of the kernel estimate and the '
         'lognormal fits better, once the\nnumber of declarations is already known. Measured on '
         'datasets the model was not fitted to. Under market-share weights\nnothing reaches a '
         'seventh of the orange line. The one large bar on the left is a property of the WEIGHTS, '
         'not of the data:\nit needs the market shares, which is exactly what a practitioner does '
         'not have.')
fig.text(0.5, -0.09, _note, ha='center', va='top', fontsize=figstyle.ANNOT_PT,
         color='#555555')
fig.tight_layout(w_pad=2.2); figstyle.check_overlaps(fig)
fig.savefig(f'{OUT}/figures/CompareUQMethods_FIG_ChoiceDrivers.png', bbox_inches='tight')
plt.show()


In [ ]:
# SUPPLEMENT: the rolling average against its replacement, kept so the two
# can be checked against each other.
#
# The left panel is the rolling mean over 250 datasets each side, exactly as
# the study drew it. The right is the same data with a bootstrap band from
# equal-count bins and a density rug. If the two tell different stories the
# difference is a finding, not an improvement.
_old_curves = pd.read_csv(f'{OUT}/tables/TABLE_MethodCurves.csv.gz')
_curves_disk = pd.read_csv(f'{OUT}/tables/TABLE_ReductionCurves.csv.gz')


def plot_old_vs_new(arm, metric, value, fname):
    how = metricred.METRIC_TRANSFORM.get(metric, 'identity')
    fig, axes = plt.subplots(1, 2, figsize=(figstyle.FULL_WIDTH, 2.7),
                             gridspec_kw=dict(wspace=0.26))
    o = _old_curves[(_old_curves.arm == arm) & (_old_curves.value == 'w1')
                    & (_old_curves.characteristic == metric)]
    for pewt in PEWT:
        p = o[o.method == pewt]
        if len(p):
            axes[0].plot(metricred.transform(p.x.to_numpy(float), how),
                         p.y_smooth, color=dct_colors[pewt], linewidth=1.2)
    axes[0].set_title('rolling mean, 250 datasets each side', fontsize=8)
    n = _curves_disk[(_curves_disk.arm == arm) & (_curves_disk.value == value)
                     & (_curves_disk.characteristic == metric)]
    for pewt in PEWT:
        pb = n[(n.kind == 'binned') & (n.method == pewt)].sort_values('x_center')
        if len(pb):
            axes[1].fill_between(pb.x_center, pb.lo, pb.hi, step='mid',
                                 color=dct_colors[pewt], alpha=0.16,
                                 linewidth=0)
        p = n[(n.kind == 'lowess') & (n.method == pewt)].sort_values('x_center')
        if len(p):
            axes[1].plot(p.x_center, p['mean'], color=dct_colors[pewt],
                         linewidth=1.2)
    axes[1].set_title('bootstrap band, and a rug where the datasets are',
                      fontsize=8)
    for ax in axes:
        ax.set_xlabel(f'{metric}, {how} scale', fontsize=7)
        ax.set_ylabel('W1', fontsize=7)
        ax.set_yscale('log')
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=6.5)
    fig.suptitle(f'Same story, with the uncertainty shown: {metric}, '
                 f'{arm} arm', fontsize=9.5, y=1.04, x=0.02, ha='left')
    plt.savefig(f'{OUT}/figures/{fname}', bbox_inches='tight', dpi=300)
    plt.show()


plot_old_vs_new('empirical', 'n', 'w1_cv',
                'CompareUQMethods_SUPP_RollingVersusBinned_Empirical.png')
plot_old_vs_new('synthetic', 'n', 'w1_market',
                'CompareUQMethods_SUPP_RollingVersusBinned_Synthetic.png')